# TRIBE v2 in-silico mapping: run the model on a free Colab GPU

Runs the whole pipeline in the cloud: stimuli (regenerated here, ~2 min) → TRIBE v2 predictions → map fitting → evaluation.
At the end you download one small zip with the results and figures.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (free tier), then *Runtime → Run all*.

* No Hugging Face token is needed: the stimuli contain no speech, so TRIBE never loads its text model (Llama).
* The video experiments are the slow part (the V-JEPA 2 video encoder runs on 64-frame clips twice per second).
  Each experiment is saved as soon as it finishes, so if the session disconnects, re-run and it continues where it stopped
  (mount Google Drive in step 2 to keep files across sessions).

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Upload the project and (optionally) keep work on Google Drive

Upload `tribe-mapping.zip` (~45 MB: the code plus example predictions) when the file picker appears.

In [ ]:
from google.colab import files
up = files.upload()                      # choose tribe-mapping.zip
!unzip -q -o tribe-mapping.zip -d /content
%cd /content/tribe-mapping

In [ ]:
# OPTIONAL: keep stimuli, predictions and the feature cache on Google Drive so a disconnect loses nothing
USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/tribe-mapping-work
    !rm -rf work && ln -s /content/drive/MyDrive/tribe-mapping-work work

## 3. Install TRIBE v2 (~3-5 min)

If the import check fails with a NumPy error, use *Runtime → Restart session* and run from this cell again.

In [ ]:
!pip install -q "tribev2 @ git+https://github.com/facebookresearch/tribev2.git" nilearn
import tribev2, torch
print("tribev2 OK, CUDA:", torch.cuda.is_available())

## 4. Reference maps and stimuli

In [ ]:
!python scripts/00_get_references.py config.yaml
!python scripts/01_make_stimuli.py config.yaml

## 5. Run TRIBE v2

Audio experiments take minutes; each video experiment can take much longer on a T4 (the progress bar shows clips per second).

In [ ]:
import os
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
!python scripts/02_run_tribe.py config.yaml --device cuda --cache work/tribe_cache --only tono_up tono_down

In [ ]:
!python scripts/02_run_tribe.py config.yaml --device cuda --cache work/tribe_cache --only hemifield motion

In [ ]:
!python scripts/02_run_tribe.py config.yaml --device cuda --cache work/tribe_cache --only retino_wedge retino_ring

## 6. Fit maps, evaluate, and download the results

In [ ]:
!python scripts/03_fit_maps.py config.yaml
!python scripts/04_evaluate.py config.yaml

In [ ]:
!cd work && zip -q -r /content/tribe_results.zip predictions_tribev2 results
files.download("/content/tribe_results.zip")